<a href="https://colab.research.google.com/github/MSA-XVIII/Driving-Safety-Chatbot-using-LLaMA-3.2-3b/blob/main/NLPAssignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate sentencepiece
!pip install -q rouge-score nltk evaluate

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1

In [ ]:
import torch
import json
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer
from evaluate import load as load_metric

In [ ]:
from datasets import load_dataset, DatasetDict

# 1. Load the PRISTINE file we generated earlier
raw_data = load_dataset("json", data_files="rto_dataset_perfect.jsonl", split="train")

# ─── Format into LLaMA 2 chat template ─────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a certified driving instructor and road safety expert. "
    "Answer questions about traffic rules, road signs, and safe driving "
    "practices accurately and concisely."
)

def format_prompt(example):
    """Wrap each example in the LLaMA 2 instruction template."""
    return {
        "text": (
            f"<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n"
            f"{example['instruction']} [/INST] "
            f"{example['response']} </s>"
        )
    }

# Build HuggingFace Dataset
dataset = raw_data.map(format_prompt)

# Train / Val / Test split  (80 : 10 : 10)
split1 = dataset.train_test_split(test_size=0.2, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict({
    "train": split1["train"],
    "validation": split2["train"],
    "test": split2["test"],
})
print(dataset_dict)
print("\nExample prompt:\n", dataset_dict["train"][0]["text"])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
MODEL_ID = "meta-llama/Llama-3.2-3B"
# Note: you need to request access at https://huggingface.co/meta-llama
# Then: huggingface-cli login  OR  set HF_TOKEN env variable

# ─── 4-bit quantization config ─────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4 — best for LLMs
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# ─── Load tokenizer ─────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token   # LLaMA 2 has no pad token by default
tokenizer.padding_side = "right"

# ─── Load base model in 4-bit ───────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print(f"Base model loaded. Parameters: {model.num_parameters():,}")

In [ ]:
# ─── LoRA configuration ─────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                        # Rank — controls adapter capacity
    lora_alpha=32,               # Scaling factor (usually 2×r)
    lora_dropout=0.05,           # Dropout on LoRA layers
    bias="none",
    target_modules=[             # Which layers to adapt (LLaMA 2 attention)
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected output: ~1–2% of total params are trainable

In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from trl import SFTTrainer
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# 1. WIPE THE MEMORY SLATE CLEAN
try:
    del model
    del trainer
except NameError:
    pass
torch.cuda.empty_cache()
gc.collect()

OUTPUT_DIR = "./llama2-driving-safety"

print("1. Loading 4-bit config...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("2. Reloading Base Model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

print("3. Manually Building LoRA Adapters...")
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# WE APPLY LORA HERE, BEFORE THE TRAINER
model = get_peft_model(model, peft_config)

print("4. Final BFLOAT16 Kill-Switch...")
# We now lock the newly created LoRA weights strictly to float16
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16 or param.requires_grad:
        param.data = param.data.to(torch.float16)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=10,
    save_steps=50,
    report_to="none",
)

print("5. Initializing Trainer...")
trainer = SFTTrainer(
    model=model,                     # Handing it the pre-built, locked model
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["validation"],
    args=training_args,
    processing_class=tokenizer,
    # CRITICAL: peft_config is REMOVED here so the trainer doesn't overwrite our fix
)

print("6. Starting training...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
print("Training complete! Model saved to", OUTPUT_DIR)

In [ ]:
import json

# 1. Read your current file
with open("rto_dataset.jsonl", "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

clean_lines = []

# 2. Clean it up line by line
for line in raw_lines:
    line = line.strip() # Removes invisible spaces and newlines

    # Skip empty lines or stray brackets (if you accidentally added [ or ])
    if not line or line == "[" or line == "]":
        continue

    # Strip trailing commas if they accidentally got added at the end of the line
    if line.endswith(","):
        line = line[:-1]

    # Verify it is actually valid JSON before keeping it
    try:
        json.loads(line)
        clean_lines.append(line)
    except Exception as e:
        print(f"Removed a broken line: {line[:50]}...")

# 3. Write out a perfect, guaranteed-to-work file
with open("rto_dataset_perfect.jsonl", "w", encoding="utf-8") as f:
    f.write("\n".join(clean_lines))

print(f"Success! Created a perfect file with {len(clean_lines)} rows.")